# <center>Big Data for Engineers &ndash; Exercise 7</center>
## <center>Spring 2026 &ndash; Week 7 &ndash; ETH Zurich</center>
## <center>Massive Parallel Processing - MapReduce</center>

## 1. MapReduce theory
### 1.1 Theory questions
1. When reading a PB-sized dataset for a map task, is it usually kept as a huge HDFS file or is it sharded? Why? Are we forced to adopt one strategy over the other?
2. Given a partitioned dataset, how can we describe the dataflow of a query executed on it? What are the extreme cases? What is the impact on the execution?
3. When is it possible to use a reduce function as accumulate function without changing the results? Give some examples.
4. What are the fundamental arguments to be passed in every map and reduce functions? What is the relationship between the data types?
5. During a MapReduce query execution, when can reduce functions be called? Why? What about accumulate functions?

### 1.2 Combine feasibility
Given the following map-reduce functions, which of these can have its reduce function also used as a combine function?
Justify your answers.

1. `key` is the constant 1, `value` is an integer
```javascript
function map(key, value):
    emit(1, value)

function reduce(key, values[]):
    emit(1, sum(values))
```

2. `key` is the constant 1, `value` is an integer
```javascript
function map(key, value):
    emit(1, value)

function reduce(key, values[]):
    emit(1, sort(values)[0])
```
3. `key` is a float, `value` is an integer
```javascript
function map(key, value):
    emit(1, value)

function reduce(key, values[]):
    emit(1, sum(values)/len(values))
```

4. `key` is a string, `value` is a string (the + operator is string concatenation)
```javascript
function map(key, value):
    emit(key, value)

function reduce(key, values[]):
    res = key
    for val in sort(values):
        res = res + val
    res = res + key
    emit(key, res)
```

### 1.3 Terminology
Give a definition of the following terms in the context of MapReduce
- function
- task
- slot
- phase

For all of these terms, specify if they can be defined for map, reduce and combine.

## 1.4 Execution simulation
In this exercise, we will simulate the execution of a MapReduce job. To start with, we introduce the initial settings:
- Two TaskTrackers with 2 CPU cores each.
- Resources partitioned as 3 map slots and 1 reduce slot.
- One million input key-value pairs, partitioned into five chunks of 100k pairs each and two chunks of 250k pairs each.
- Each call to a map function takes 0.1ms.
- Each call to a reduce function takes 0.1ms for each value.
- Moving an intermediate key-value pair during shuffling takes 0.1ms.

We want to calculate the total runtime.

### Map phase
The job tracker assigns the first tasks to the three slots in this way:
- map slot1: task1, 100k
- map slot2: task2, 250k
- map slot3: task3, 250k

Left: four tasks of 100k.
    
The remaining tasks are assigned to a slot as soon as it finishes the current task. If two slots finish at the same time, the task is assigned to the one with lower ID.
Your aim is to calculate the schedule of each map slot.
    
You can keep track of the schedule by extending this table:
| Slot  | T = 0ms |
|-------|---------|
| Slot1 | t1:100k |
| Slot2 | t2:250k |
| Slot3 | t3:250k |

### Shuffling phase
The map tasks have output two key-value pairs for each key-value pair of input (one every 0.05ms).
Remember that shuffling can proceed in parallel with the map phase. Assume that as soon as a map function emits a pair, this can be shuffled.
Assume that inside a slot the intermediate pairs are moved sequentially.
    
When does the shuffling phase terminate? You can round the runtime to the millisecond.
Does it matter if a map slot resides on the same node with a reduce slot?

### Reduce phase
Once the shuffling phase has terminated, the reduce phase can begin. There are two million key value pairs to process.

What is the total runtime of the MapReduce job?

### 1.5 Physical architecture
MapReduce V1 has two main process types: **JobTracker** and **TaskTracker**. Insert them in the table below using the parallelism with HBase and HDFS
| Master     | Worker       |
|------------|--------------|
| NameNode   | DataNode     |
| HMaster    | RegionServer |
| - | - |

<img src="https://polybox.ethz.ch/index.php/s/KtMNzick92GE44y/download" width="30%">

Why do we want these two processes to run on specific nodes?

## 2 MapReduce practice
### 2.1 Abstract functions
In this session, we will implement different map-reduce jobs using Python with a syntax that closely matches the pseudocode.
You can define a map task as a function that takes a key, a value and a context object as inputs and emits output pairs using
the `emit` method of the context object.
To test your solutions, you can execute the task using `submit(data, mapper, reducer)` where `data` is a list of key-value
pairs, `mapper` is the map function and `reducer` is the reduce function.

In [ ]:
from collections import defaultdict

class Context:
    def __init__(self):
        self.pairs = []

    def merge(self):
        merged = defaultdict(list)
        for key, value in self.pairs:
            merged[key].append(value)
        return merged.items()

    def emit(self, key, value):
        self.pairs.append((key, value))
        

def submit(pairs, mapper, reducer):
    map_context = Context()
    reduce_context = Context()
    for key, value in pairs:
        mapper(key, value, map_context)
    
    for key, values in map_context.merge():
        reducer(key, values, reduce_context)

    return reduce_context.pairs

In [ ]:
# Example usage: count the occurrencies of each key
def mapper(key, value, ctx):
    ctx.emit(key, None)

def reducer(key, values, ctx):
    ctx.emit(key, len(values))

data = [('a', None), ('a', None), ('b', None)]
submit(data, mapper, reducer)

In [ ]:
# 1. Return the unique keys of the input key-value pairs as [(key1, 1), ..., (keyN, 1)]

def mapper(key, value, ctx):
    ...

def reducer(key, values, ctx):
    ...

In [ ]:
# 2. Group by key and compute max, min, avg of the values as (key, (max, min, avg))

def mapper(key, value, ctx):
    ...

def reducer(key, values, ctx):
    ...

In [ ]:
# 3. count the number of words in a document with length above 5. A document might be chunked into several pairs
#    You can consider the space " " as the only word delimiter.
#    Input: [(doc_id, text),...]
#    Output: [(doc_id, n_words),]

def mapper(key, value, ctx):
    ...

def reducer(key, values, ctx):
    ...

In [ ]:
# 4. Count the occurencies of letters (case insensitive) in a document (0 if not occurring), sorted alphabetically (a document might come in chunks)
#    Hint: the builtin function ord(string) returns the ASCII number of string if string is a single character
#          and chr(integet) returns the corresponding char.
#    Input: [(doc_id, text),...]
#    Output: [(doc_id, [n_a, ..., n_z]),]

def mapper(key, value, ctx):
    ...

def reducer(key, values, ctx):
    ...

In [ ]:
# 5. Compute the scalar product of two vectors (hint: you could need more than one map-reduce job)
#    Input: [(position, value), ...]. The input sequence contains both vectors, encoded as pairs (i, v_i).
#    Output: [(1, A * B)]

def mapper(key, value, ctx):
    ...

def reducer(key, values, ctx):
    ...

### 2.2 Reverse engineering
Imagine we have the following table `students`:
| student_id | first_name  | last_name | grade |
|------------|------------|-----------|-------|
| AAAA       | Ada        | Lovelace  | 8     |
| AAAB       | Alan       | Turing    | 7     |
| AAAC       | Haskell    | Curry     | 9     |
...

Imagine it is stored as JSON Line and it is fed into MapReduce as a sequence like `(student_id, {'first_name':name, 'last_name':surname, 'grade':grade})`.

Given the pseudocode of map and reduce functions, infer the equivalent SQL statement

1.
```javascript
function map(key, value):
    if value["first_name"] == "Ada":
        emit(1, key)

function reduce(key, values[]):
    emit(1, values)
```

2.
```javascript
function map(key, value):
    emit(value["grade"], 1)

function reduce(key, values[]):
    emit(key, len(values))
```

3.
```javascript
function map(key, value):
    emit(key, value["grade"])

function reduce(key, values[]):
    c = sum(values)/len(values)
    if c < 5:
        emit(key, c)
```

4.
```javascript
function map(key, value):
    emit(value["first_name"], 1)

function reduce(key, values[]):
    emit(key, 1))
```

5.
```javascript
function map(key, value):
    emit(key, value["grade"])

function reduce1(key, values[]):
    emit(1, sum(values)/len(values))

function reduce2(key, values[]):
    emit(1, max(values))
```

Call chain: `map -> reduce1 -> reduce2`.